<a href="https://colab.research.google.com/github/DiyaMalik07/SC_Project/blob/main/MemeticAlgorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pymoo

import numpy as np
import pandas as pd
import time
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.core.callback import Callback
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.operators.crossover.hux import HalfUniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize import minimize

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Libraries imported successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 1.4 MB/s eta 0:00:00
Libraries imported successfully!


In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/internet_ads/ad.data"
urllib.request.urlretrieve(url, "ad.data")

df = pd.read_csv("ad.data", header=None, low_memory=False)
print(f"Dataset shape (rows, columns): {df.shape}")

print("Preprocessing data")
df = df.replace(r'^\s*\?\s*$', np.nan, regex=True)

label_col = df.columns[-1]
y = df[label_col].astype(str).str.strip().apply(
    lambda v: 1 if v.lower().startswith('ad') else 0
).values

X = df.drop(columns=[label_col])
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.mean())
X = X.values

n_features = X.shape[1]
print(f"Total features: {n_features}")
print(f"Total instances: {X.shape[0]}")
print(f"Class balance -> ad: {sum(y==1)}  |  nonad: {sum(y==0)}")

Dataset shape (rows, columns): (3279, 1559)
Preprocessing data
Total features: 1558
Total instances: 3279
Class balance -> ad: 459  |  nonad: 2820


In [3]:
print("Splitting data (Train 60% / Validation 20% / Test 20%)")

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Train samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Splitting data (Train 60% / Validation 20% / Test 20%)
Train samples: 1967
Validation samples: 656
Test samples: 656


In [4]:
print("Training Random Forest on ALL features")

start_time = time.time()
baseline_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
baseline_model.fit(X_train, y_train)
baseline_train_time = time.time() - start_time

baseline_predictions = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_f1 = f1_score(y_test, baseline_predictions)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")
print(f"Baseline F1-score : {baseline_f1:.4f}")
print(f"Features used     : {n_features}")
print(f"Training time     : {baseline_train_time:.2f} seconds")

Training Random Forest on ALL features
Baseline Accuracy : 0.9665
Baseline F1-score : 0.8690
Features used     : 1558
Training time     : 4.08 seconds


In [5]:
print("Setting up Memetic Genetic Algorithm (MGA)")

class DiverseBinarySampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        population = np.zeros((n_samples, problem.n_var), dtype=bool)
        for i in range(n_samples):
            selection_prob = np.random.uniform(0.02, 0.80)
            population[i] = np.random.rand(problem.n_var) < selection_prob
            if population[i].sum() == 0:
                population[i, np.random.randint(0, problem.n_var)] = True
        return population

class MGAFeatureSelectionProblem(Problem):
    def __init__(self, X_train, y_train, X_val, y_val, alpha=0.95):
        super().__init__(n_var=n_features, n_obj=1, n_constr=0, xl=0, xu=1, vtype=bool)
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.alpha = alpha

    def _evaluate(self, population, out, *args, **kwargs):
        fitness_list = []
        for individual in population:
            mask = individual.astype(bool)
            if mask.sum() == 0:
                mask[np.random.randint(0, self.n_var)] = True

            model = RandomForestClassifier(n_estimators=30, random_state=RANDOM_SEED, n_jobs=-1)
            model.fit(self.X_train[:, mask], self.y_train)
            predictions = model.predict(self.X_val[:, mask])

            f1 = f1_score(self.y_val, predictions)
            fitness = (self.alpha * (1.0 - f1)) + ((1.0 - self.alpha) * (mask.sum() / self.n_var))
            fitness_list.append(fitness)
        out["F"] = np.column_stack([fitness_list])

class MemeticLocalSearchCallback(Callback):
    def __init__(self, n_neighbors=3):
        super().__init__()
        self.n_neighbors = n_neighbors
    def notify(self, algorithm):
        problem = algorithm.problem
        pop = algorithm.pop

        F_vals = pop.get("F")
        if F_vals is None or len(F_vals) == 0:
            return

        best_idx = np.argmin(F_vals)
        best_ind = pop[best_idx]

        best_X = best_ind.get("X").astype(bool)
        best_F = F_vals[best_idx][0]

        neighbors_X = []
        for _ in range(self.n_neighbors):
            neighbor = best_X.copy()
            flip_idx = np.random.randint(0, problem.n_var)
            neighbor[flip_idx] = not neighbor[flip_idx]

            if neighbor.sum() == 0:
                neighbor[np.random.randint(0, problem.n_var)] = True
            neighbors_X.append(neighbor)

        neighbors_X = np.array(neighbors_X)

        out = {}
        problem._evaluate(neighbors_X, out)
        neighbors_F = out["F"]

        best_neighbor_idx = np.argmin(neighbors_F)
        if neighbors_F[best_neighbor_idx][0] < best_F:
            best_ind.set("X", neighbors_X[best_neighbor_idx])
            best_ind.set("F", neighbors_F[best_neighbor_idx])

Setting up Memetic Genetic Algorithm (MGA)


In [6]:
POPULATION_SIZE = 40
GENERATIONS = 30

mga_problem = MGAFeatureSelectionProblem(X_train, y_train, X_val, y_val, alpha=0.95)

mga_algorithm = GA(
    pop_size=POPULATION_SIZE,
    sampling=DiverseBinarySampling(),
    crossover=HalfUniformCrossover(),
    mutation=BitflipMutation(prob=5.0 / n_features),
    eliminate_duplicates=True
)

print(f"Running Memetic GA optimization for {GENERATIONS} generations")
start_time = time.time()

mga_result = minimize(
    mga_problem,
    mga_algorithm,
    termination=('n_gen', GENERATIONS),
    callback=MemeticLocalSearchCallback(n_neighbors=3),
    seed=RANDOM_SEED,
    verbose=True
)

mga_run_time = time.time() - start_time
print(f"\nMGA finished in {mga_run_time:.2f} seconds")

Running Memetic GA optimization for 30 generations
n_gen  |  n_eval  |     f_avg     |     f_min    
     1 |       40 |  0.1582475666 |  0.0722278598
     2 |       80 |  0.0951430745 |  0.0722278598
     3 |      120 |  0.0848879336 |  0.0691477678
     4 |      160 |  0.0798731820 |  0.0654589398
     5 |      200 |  0.0753824471 |  0.0617657289
     6 |      240 |  0.0721291565 |  0.0616373592
     7 |      280 |  0.0687298332 |  0.0552590343
     8 |      320 |  0.0678026586 |  0.0552590343
     9 |      360 |  0.0663856975 |  0.0552590343
    10 |      400 |  0.0641624635 |  0.0552590343
    11 |      440 |  0.0630593234 |  0.0552590343
    12 |      480 |  0.0625075002 |  0.0552590343
    13 |      520 |  0.0620749085 |  0.0552590343
    14 |      560 |  0.0616563709 |  0.0552590343
    15 |      600 |  0.0616256398 |  0.0552590343
    16 |      640 |  0.0615591509 |  0.0552590343
    17 |      680 |  0.0611717571 |  0.0552590343
    18 |      720 |  0.0606322184 |  0.0552590343

In [7]:
print("Evaluating best feature subset found by Memetic GA\n")

mga_best_mask = mga_result.X.astype(bool)
mga_features_count = int(mga_best_mask.sum())

start_time = time.time()
mga_final_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
mga_final_model.fit(X_train[:, mga_best_mask], y_train)
mga_final_train_time = time.time() - start_time

mga_predictions = mga_final_model.predict(X_test[:, mga_best_mask])
mga_accuracy = accuracy_score(y_test, mga_predictions)
mga_f1 = f1_score(y_test, mga_predictions)

print("FINAL RESULTS: BASELINE vs MEMETIC GA")
print(f"{'Metric':<28}{'All Features':<20}{'MGA-Selected':<18}")
print(f"{'Accuracy':<28}{baseline_accuracy:<20.4f}{mga_accuracy:<18.4f}")
print(f"{'F1-score':<28}{baseline_f1:<20.4f}{mga_f1:<18.4f}")
print(f"{'Number of Features':<28}{n_features:<20}{mga_features_count:<18}")

reduction_pct = (1 - mga_features_count / n_features) * 100
print(f"{'Feature Reduction %':<28}{'-':<20}{reduction_pct:<18.1f}")
print(f"{'Search Algorithm Time (s)':<28}{'-':<20}{mga_run_time:<18.2f}")
print(f"{'Model Training Time (s)':<28}{baseline_train_time:<20.2f}{mga_final_train_time:<18.2f}")

Evaluating best feature subset found by Memetic GA

FINAL RESULTS: BASELINE vs MEMETIC GA
Metric                      All Features        MGA-Selected      
Accuracy                    0.9665              0.9649            
F1-score                    0.8690              0.8639            
Number of Features          1558                803               
Feature Reduction %         -                   48.5              
Search Algorithm Time (s)   -                   595.33            
Model Training Time (s)     4.08                0.82              
